In [1]:
!nvidia-smi

Wed Jul 15 10:40:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers librosa soundfile accelerate

In [3]:
from google.colab import drive;

In [4]:
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
!ls "/content/drive/MyDrive/ContextAI"

ls: cannot access '/content/drive/MyDrive/ContextAI': No such file or directory


In [6]:
!ls "/content/drive/MyDrive"

 AI_Cybersecurity_Presentation.gslides
 ArabTech_Website_Documentation.md.gdoc
 backend_document.gdoc
 Backend_Document.gdoc
'bkc-hhzj-tkg (2026-05-25 20:59 GMT+5).gvid'
'Context AI'
 Downloads.zip
 GlowWell_Skincare_Presentation.gslides
'LabSearch_Backend_Document (1).gdoc'
 LabSearch_Backend_Document.gdoc
 LabSearch_Backend_Spec_v1.0.gdoc
 LabSearch_Frontend_Design_Spec.gdoc
 LabSearch_Pakistan_SRS_v1.0.gdoc
 Material
'M.Huzaifa Mustafa CV.gdoc'
 ShopSearch_SRS_v1.1.gdoc
'STT v2 7Day Plan.gsheet'
'Team 3 test case.gdoc'
'Technical Spec Review Developer Guidelines.gdoc'
'Urdu Model Validation Checklist.gsheet'
'Urdu Whisper Banking IVR Plan.gsheet'
 Web_Development_Roadmap_2025.gdoc
 WebDev_Roadmap_2026.gdoc


In [7]:
!ls "/content/drive/MyDrive/Context AI"

processed


In [8]:
!ls "/content/drive/MyDrive/Context AI/processed"

clean  errors.log  manifest.jsonl  pipeline.log  segments  transcription.log


In [9]:
import os, re, json, time
import torch
import librosa
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# --- paths (Google Drive) ---
BASE = "/content/drive/MyDrive/Context AI/processed"
MANIFEST_PATH = f"{BASE}/manifest.jsonl"
SEGMENTS_DIR = f"{BASE}/segments"
TRANSCRIPT_LOG = f"{BASE}/transcription.log"

HF_ASR_MODEL_ID = "ihanif/whisper-medium-urdu"
TARGET_SR = 16000

# --- device ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# --- mixed-script heuristic (same as local code) ---
_LATIN = re.compile(r"[A-Za-z]")
_URDU = re.compile(r"[\u0600-\u06FF]")
def _is_mixed_script(text):
    return bool(_LATIN.search(text)) and bool(_URDU.search(text))

def _needs_retranscript(record):
    t = record.get("transcript")
    if t is None or t == "":
        return True
    return _is_mixed_script(t)

# --- load manifest ---
def load_manifest():
    records = []
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def save_manifest(records):
    tmp = MANIFEST_PATH + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    os.replace(tmp, MANIFEST_PATH)

records = load_manifest()

# --- segment selection: same rules as local pipeline ---
pending = []
n_skipped_verified = 0
n_unchanged = 0
for r in records:
    if r.get("verified"):
        n_skipped_verified += 1
        continue
    if _needs_retranscript(r):
        pending.append(r)
    else:
        n_unchanged += 1

total = len(pending)
print(f"Segments total={len(records)} | verified(skipped)={n_skipped_verified} | to-process={total} | unchanged={n_unchanged}")

# --- load model onto GPU ---
print("Loading model...")
processor = WhisperProcessor.from_pretrained(HF_ASR_MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(HF_ASR_MODEL_ID, torch_dtype="float32")
model = model.to(device)

forced = processor.get_decoder_prompt_ids(language="urdu", task="transcribe")
decoder_input_ids_base = torch.tensor([[i for _, i in forced]], dtype=torch.long).to(device)
print("Model loaded on", device)

def transcribe_one(wav_path):
    y, _ = librosa.load(wav_path, sr=TARGET_SR, mono=True)
    inputs = processor(y, sampling_rate=TARGET_SR, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        generated = model.generate(
            **inputs,
            decoder_input_ids=decoder_input_ids_base,
            num_beams=1,
            max_new_tokens=225,
        )
    cont = generated[0][decoder_input_ids_base.shape[1]:]
    text = processor.decode(cont, skip_special_tokens=True)
    text = re.sub(r"<\|[^|]*\|>", "", text)
    return text.strip()

# --- main loop ---
done = 0
n_reprocessed = 0
n_errors = 0
start = time.time()

with open(TRANSCRIPT_LOG, "a", encoding="utf-8") as lf:
    lf.write(f"[Colab GPU run started] 0/{total} segments\n")

for r in pending:
    seg_id = r.get("segment_id")
    fname = r.get("filename")
    source_type = r.get("source_type", "")
    wav = f"{SEGMENTS_DIR}/{source_type}/{fname}"
    if not os.path.exists(wav):
        wav = f"{SEGMENTS_DIR}/{fname}"
    if not os.path.exists(wav):
        print(f"MISSING: {fname}")
        n_errors += 1
        done += 1
        continue
    try:
        text = transcribe_one(wav)
        r["transcript"] = text
        r["transcript_model"] = HF_ASR_MODEL_ID
        n_reprocessed += 1
    except Exception as e:
        print(f"ERROR on {seg_id}: {e}")
        n_errors += 1

    done += 1
    if done % 10 == 0 or done == total:
        save_manifest(records)
        elapsed = time.time() - start
        rate = done / elapsed if elapsed else 0
        eta = (total - done) / rate / 60 if rate else 0
        msg = f"{done}/{total} done ({100*done/total:.1f}%) | {rate:.2f} seg/s | ETA {eta:.1f} min"
        print(msg)
        with open(TRANSCRIPT_LOG, "a", encoding="utf-8") as lf:
            lf.write(msg + "\n")

save_manifest(records)
print(f"DONE. reprocessed={n_reprocessed} errors={n_errors} skipped_verified={n_skipped_verified} unchanged={n_unchanged}")

Using device: cuda
Segments total=903 | verified(skipped)=28 | to-process=239 | unchanged=636
Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/830 [00:00<?, ?B/s]

...

220/239 done (92.1%) | 0.05 seg/s | ETA 5.8 min


Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


230/239 done (96.2%) | 0.05 seg/s | ETA 2.8 min


Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


239/239 done (100.0%) | 0.05 seg/s | ETA 0.0 min
DONE. reprocessed=239 errors=0 skipped_verified=28 unchanged=636
